In [23]:
## ------- edit params first here --------------

INPUTROOT = 'export_c3/'
OUTPUTROOT = 'report_c3/'

# ------------------------------------------------

# utils for os.path
from os.path import join as J
from os.path import exists as E
from os import listdir as L
import pandas as pd
import numpy as np
from typing import Literal
pd.set_option('display.max_rows',6)
print('global L,E,J is defined')

#---------------- setup PATHs -------------------
import os
WORKDIR = os.getcwd()

## setup OUTPUTPATH and WOKER
import sys

#---------------- load librarys -------------------

def write_excel(d,fname,outputdir='report',force_ext:None|Literal['xls','xlsx']=None,max_row=50_000,index=True):
    """
    :engine: 'xlsxwriter' for .xlsx, 'xlwt' for .xls
    ext will be auto determined by the fname when force_ext is None
        if not provide in fname, default is .xlsx
    :max_row: max row number for each sheet, if exceed, skip
    """
    _ = fname.split('.')
    if len(_) > 1:
        ext = _[-1]
        name = ".".join(_[:-1])
    else:
        name = fname
        ext = 'xlsx'
    if force_ext is not None:
        assert type(force_ext) == str, "force_ext must be in ['xls','xlsx']"
        ext = force_ext
    if ext == 'xlsx':
        engine = 'xlsxwriter'
    elif ext == 'xls':
        engine = 'xlwt'
    else:
        raise Exception(f'auto deteminded ext = {ext}, this is not supported, mainly due to there is dot (.) in your current fname,  please manually set ext to .xlsx or .xls')
    out = J(outputdir, name + '.' + ext)

    with pd.ExcelWriter(out, engine=engine) as writer:
        EXCEL_SHEETS_LIMIT = 255
        count = 0
        sheet_names_used = {}
        l = len(d.keys())
        if  l < EXCEL_SHEETS_LIMIT:
            temp = f'{{:0{len(str(l))}d}}'
        else:
            raise  ValueError(f'too many DataFrames({l}) to be written, maximal 255!')
        for name,df in d.items():
            tp = type(df)
            if tp != pd.DataFrame:
                try:
                    df = pd.DataFrame(df)
                    print(f'covert {name}: {tp} -> {type(df)}')
                except Exception as e:
                    print(f'skip {name}, illegal type {type(df)}')
                    continue
            l = len(df) 
            if l <= max_row:
                count  = count + 1
                cc = temp.format(count)
                print(f'  append #{cc} Sheet [{name}]')
                if len(name) > 30:
                    name = name[:28]
                if name in sheet_names_used.keys():
                    suffix = sheet_names_used[name]
                    if suffix > 100:
                        print(f'[ERR] too many(>100) sheets with the same name! skip sheet {name} : {suffix}')
                    name = name + str(suffix)
                    sheet_names_used[name] += 1
                else:
                    sheet_names_used[name] = 0
                # change any df column with dtype of datetime64+time zone to str, as current xlwt does not support datetime with time zone info format
                mask = pd.Series([pd.api.types.is_datetime64_any_dtype(i) for i in df.dtypes])
                df = df.astype({i:str for i in df.columns[mask]})
                df.to_excel(writer,sheet_name=name,index=index)
            else:
                print(f'⛔ skip [{name}.dfx] for it contains {l} rows exceeding max_row setting.')
#     writer.save()
    print(f'writing 1 .xls @ {out}')



global L,E,J is defined


### pre wash xlsx -> dfx

In [10]:
### new ones @2025-03-28

#C1
df = pd.read_excel('external_data/Cmap_drug_screen/C1-query_result.gct.xlsx',na_values=['','-666',-666])
df.columns = df.columns.str.replace(r'[^\x00-\x7F]',"",regex=True)
df = df[df['pert_type'] == 'trt_cp']
df = df[~df['moa'].isna()]

out = 'external_data/Cmap_drug_screen/C1_trt_cp.dfx'
df.to_csv(out,sep='\t',index=False)
print(f"writing 1 file to\n    {out}")


writing 1 file to
    external_data/Cmap_drug_screen/C1_trt_cp.dfx


In [9]:
#C2
df = pd.read_excel('external_data/Cmap_drug_screen/C2-query_result.gct.xlsx',na_values=['','-666',-666])
df.columns = df.columns.str.replace(r'[^\x00-\x7F]',"",regex=True)
df = df[df['pert_type'] == 'trt_cp']
df = df[~df['moa'].isna()]

out = 'external_data/Cmap_drug_screen/C2_trt_cp.dfx'
df.to_csv(out,sep='\t',index=False)
print(f"writing 1 file to\n    {out}")


writing 1 file to
    external_data/Cmap_drug_screen/C2_trt_cp.dfx


### load ALL

In [13]:
todo = [i for i in L('external_data/Cmap_drug_screen/') if i[-4:]=='.dfx']
todo = pd.Series(todo).sort_values()
todo


0    C1_trt_cp.dfx
1    C2_trt_cp.dfx
dtype: object

In [18]:
for dfx in todo:
    df_ori = pd.read_csv(f'external_data/Cmap_drug_screen/{dfx}',sep='\t').reset_index(drop=True)
    df = df_ori
    df = df.rename(columns={'fdr_q_nlog10pp':'fdr_q_nlog10'})
    break

In [19]:
# Palbociclib (PD-0332991)HCL
# Ribociclib (LEE011)
# Ipatasertib(GDC-0068)
# Capivasertib (AZD5363)
# Talazoparib(BMN 673)
# Olaparib (AZD2281)
# I-BET151 (GSK1210151A)
# (+)-JQ1
# Docetaxel (RP56976)
# Enzalutamide
# Darolutamide (ODM-201)
# Apalutamide (ARN-509)
# Carboplatin (NSC 241240)
# Etoposide (VP-16)
# GSK503
# GSK343
# Trametinib (GSK1120212)
# Cobimetinib (GDC-0973)
# Alisertib (MLN8237)
# Pulrodemstat (CC-90011) besylate
# Erdafitinib (JNJ-42756493)
# Docetaxel (RP56976)
# Olaparib (AZD2281)
# Ruxolitinib (INCB018424)

need = ["Palbociclib","(PD-0332991)","Ribociclib","(LEE011)","Ipatasertib",
 "(GDC-0068)","Capivasertib","(AZD5363)","Talazoparib","(BMN673)",
 "I-BET151","(GSK1210151A)","Docetaxel",
 "(RP56976)","Enzalutamide","Darolutamide","(ODM-201)","Apalutamide",
 "(ARN-509)","Carboplatin","(NSC241240)","Etoposide","(VP-16)",
 "GSK503","GSK343","Trametinib","(GSK1120212)","Cobimetinib",
 "(GDC-0973)","Alisertib","(MLN8237)","Pulrodemstat","(CC-90011)besylate",
 "Erdafitinib","(JNJ-42756493)","Docetaxel","(RP56976)",
 "Olaparib","(AZD2281)","Ruxolitinib","(INCB018424)"]



a = df_ori['pert_iname'].drop_duplicates()
a = a[~a.isna()]
a = a.astype(str)
a2 = pd.Series([i.replace('-',"").upper() for i in a.values],index=a.index)    

a2

res = []
for i in need:
    fd = a2[a2.str.contains(i.replace('-',"").upper())]
    if len(fd)>0:
        res.append(fd)
res = pd.concat(res)
fd = a[a.index.isin(res.index)]
FD = fd
fd

/tmp/ipykernel_1618/2023505635.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  fd = a2[a2.str.contains(i.replace('-',"").upper())]


276        docetaxel
287      palbociclib
557         GDC-0068
            ...     
10827       AZD-5363
13736      alisertib
14596      I-BET-151
Name: pert_iname, Length: 15, dtype: object

In [21]:
res = {}
for dfx in todo:
    name = dfx.split('-')[0]
    df_ori = pd.read_csv(f'external_data/Cmap_drug_screen/{dfx}',na_values = '-666', sep='\t').reset_index(drop=True)
    df = df_ori
    df = df.rename(columns={'fdr_q_nlog10pp':'fdr_q_nlog10','pert_typetrt_cp':'pert_type','moa666':'moa'})
    
    df['pert_iname'] = df['pert_iname'].astype(str)

    df = df[(df['norm_cs']<0) &(df['pert_type']=='trt_cp') & ((df['fdr_q_nlog10'] > -np.log10(0.05)) | (df['pert_iname'].isin(fd)))]

    mean = df.groupby(['pert_iname'],as_index=False).agg({'norm_cs':np.mean,'fdr_q_nlog10':np.mean}).sort_values('norm_cs').reset_index(drop=True)
    mean['type'] = ''
    mean.loc[mean['pert_iname'].isin(FD),'type'] = 'sel'
    
    x = df[['pert_iname','moa','cell_iname','target_name']].fillna('').drop_duplicates().groupby('pert_iname').agg(lambda col:", ".join(col.unique())).reset_index()
    mean = pd.merge(mean,x,on = 'pert_iname',how='left',suffixes=['','__xx'])
    
    mean.index.name = name
    display(mean)
    res[name] = mean
    
    

,pert_iname,norm_cs,fdr_q_nlog10,type,moa,cell_iname,target_name
C1_trt_cp.dfx,,,,,,,
0,BRD-K68202742,-2.359400,15.653600,,HDAC inhibitor|CDK expression enhancer|ID1 exp...,VCAP,HDAC1|HDAC3|HDAC4|HDAC6|HDAC7|HDAC8|HDAC10|HDA...
1,panobinostat,-2.343100,15.653600,,HDAC inhibitor,VCAP,HDAC1|HDAC2|HDAC3|HDAC4|HDAC6|HDAC7|HDAC8|HDAC9
2,PCI-24781,-2.324000,15.653600,,HDAC inhibitor,VCAP,HDAC1|HDAC2|HDAC3|HDAC6|HDAC10|HDAC8
...,...,...,...,...,...,...,...
1579,talazoparib,-0.931450,0.254150,sel,PARP inhibitor,"A549, HELA, HT29, MCF10A, A375, HA1E",PARP2|PARP1
1580,GDC-0068,-0.928167,0.300892,sel,AKT inhibitor,"TMD8, NALM6, HEPG2, YAPC, U2OS, SKL, MINO, HEK...",AKT1|AKT2|AKT3|CYP3A5|PRKG1
1581,S-ruxolitinib,-0.796850,0.126975,sel,JAK inhibitor,"HA1E, MCF7, A375, YAPC",


,pert_iname,norm_cs,fdr_q_nlog10,type,moa,cell_iname,target_name
C2_trt_cp.dfx,,,,,,,
0,LY-2886721,-2.240300,15.653600,,Beta secretase inhibitor,A549,BACE1|BACE2
1,ftorafur,-2.141600,15.653600,,Thymidylate synthase inhibitor,U2OS,TYMS
2,erythromycin-ethylsuccinate,-2.079700,15.653600,,Cytochrome P450 inhibitor|Motilin receptor ago...,HCC515,CYP3A4|MLNR|ABCB1|ALB|CYP51A1|KCNH2|SLC47A1
...,...,...,...,...,...,...,...
1878,olaparib,-0.872577,0.293677,sel,PARP inhibitor,"HS578T, MCF10A, NPC, NCIH2073, HT29, NCIH596, ...",PARP1|PARP2|PARP3
1879,alisertib,-0.862550,0.296540,sel,Aurora kinase inhibitor,"MCF7, A375, PC3, MDAMB231, YAPC, HT29, U2OS, T...",AURKA
1880,enzalutamide,-0.840623,0.263423,sel,Androgen receptor antagonist,"A549, HCC515, HEPG2, MCF10A, HELA, NCIH508, U9...",AR|CYP2C19|CYP2C8|CYP3A4


### dump

In [24]:
write_excel(res, '_CMAP_DRUGS',outputdir='external_data/Cmap_drug_screen/')

  append #1 Sheet [C1_trt_cp.dfx]
  append #2 Sheet [C2_trt_cp.dfx]
writing 1 .xls @ external_data/Cmap_drug_screen/_CMAP_DRUGS.xlsx
